# 딥러닝 모델 실험

이 노트북은 딥러닝 모델 실험을 사용자가 직접 실행하고 검증하기 위한 노트북입니다.

중요 기준:

- Logistic Regression과 Random Forest는 전통 ML 기준선(reference baseline)입니다.
- 이 노트북은 같은 tensor split에서 MLP, SimpleRNN, LSTM, GRU, BiLSTM, 1D CNN을 비교합니다.
- 기본값은 `RUN_FULL_EXPERIMENTS = False`입니다. 전체 학습 그리드가 실수로 바로 실행되지 않게 하기 위한 안전장치입니다.
- 먼저 smoke test를 실행하세요. 정상 동작을 확인한 뒤 `RUN_FULL_EXPERIMENTS = True`로 바꾸고 전체 실험을 실행하면 됩니다.

입력 데이터:

```text
data/processed/deep_learning_sequences/window_7/
data/processed/deep_learning_sequences/window_14/
data/processed/deep_learning_sequences/window_30/
```


In [20]:
# 1. 기본 설정
# 이 셀부터 아래 방향으로 순서대로 실행하세요.
# PyTorch가 없다면 현재 사용 중인 가상환경에 필요한 패키지를 먼저 설치하세요.
# 예시: pip install torch scikit-learn pandas numpy matplotlib


from __future__ import annotations

import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEQUENCE_ROOT = PROJECT_ROOT / "data" / "processed" / "deep_learning_sequences"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "deep_learning_experiments"
MODEL_DIR = PROJECT_ROOT / "models" / "deep_learning"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUN_FULL_EXPERIMENTS = True

RUN_SMOKE_TEST = True

WINDOWS = [7, 14, 30]
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 6
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SEQUENCE_ROOT exists:", SEQUENCE_ROOT.exists())
print("DEVICE:", DEVICE)


PROJECT_ROOT: c:\workSpace\DeepLearnin_sleep
SEQUENCE_ROOT exists: True
DEVICE: cpu


In [4]:
# 2. 재현성 설정
# 가능한 범위에서 반복 실행 결과가 비슷하게 나오도록 seed를 고정합니다.


def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_STATE)


In [5]:
# 3. Tensor 로딩 함수
# 각 .npz 파일에는 모델 입력, target label, participant/date metadata가 들어 있습니다.


SPLITS = ["train", "validation", "test"]


def load_split_npz(window: int, split: str) -> dict:
    path = SEQUENCE_ROOT / f"window_{window}" / f"{split}.npz"
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as data:
        return {key: data[key] for key in data.files}


def load_window_data(window: int) -> Dict[str, dict]:
    return {split: load_split_npz(window, split) for split in SPLITS}


def summarize_window(window: int) -> pd.DataFrame:
    rows = []
    for split, data in load_window_data(window).items():
        y = data["y"]
        rows.append(
            {
                "window": window,
                "split": split,
                "X_sequence_shape": tuple(data["X_sequence"].shape),
                "X_mlp_flattened_shape": tuple(data["X_mlp_flattened"].shape),
                "X_mlp_current_day_shape": tuple(data["X_mlp_current_day"].shape),
                "y_shape": tuple(y.shape),
                "target_mean": float(y.mean()) if len(y) else np.nan,
                "participants": len(set(data["participant_object_id"].astype(str))),
            }
        )
    return pd.DataFrame(rows)

pd.concat([summarize_window(window) for window in WINDOWS], ignore_index=True)


,window,split,X_sequence_shape,X_mlp_flattened_shape,X_mlp_current_day_shape,y_shape,target_mean,participants
0,7,train,"(1393, 7, 197)","(1393, 1379)","(1393, 197)","(1393,)",0.403446,35
1,7,validation,"(269, 7, 197)","(269, 1379)","(269, 197)","(269,)",0.394052,9
2,7,test,"(314, 7, 197)","(314, 1379)","(314, 197)","(314,)",0.420382,9
3,14,train,"(933, 14, 197)","(933, 2758)","(933, 197)","(933,)",0.382637,27
4,14,validation,"(146, 14, 197)","(146, 2758)","(146, 197)","(146,)",0.349315,7
5,14,test,"(214, 14, 197)","(214, 2758)","(214, 197)","(214,)",0.457944,5
6,30,train,"(435, 30, 197)","(435, 5910)","(435, 197)","(435,)",0.340230,19
7,30,validation,"(55, 30, 197)","(55, 5910)","(55, 197)","(55,)",0.236364,3
8,30,test,"(106, 30, 197)","(106, 5910)","(106, 197)","(106,)",0.594340,4


In [6]:
# 4. Split leakage 확인
# 같은 participant가 train, validation, test에 동시에 들어가면 안 됩니다.



def check_participant_overlap(window: int) -> None:
    data = load_window_data(window)
    participant_sets = {
        split: set(data[split]["participant_object_id"].astype(str))
        for split in SPLITS
    }
    for left_idx, left in enumerate(SPLITS):
        for right in SPLITS[left_idx + 1 :]:
            overlap = participant_sets[left] & participant_sets[right]
            if overlap:
                raise RuntimeError(
                    f"window={window}, participant overlap between {left} and {right}: {sorted(overlap)[:5]}"
                )
    print(f"window={window}: participant overlap OK")

for window in WINDOWS:
    check_participant_overlap(window)


window=7: participant overlap OK
window=14: participant overlap OK
window=30: participant overlap OK


In [7]:
# 5. DataLoader 생성 함수
# model_input 옵션은 current_day, flattened, sequence입니다.



def make_loader(
    arrays: dict,
    model_input: str,
    batch_size: int = BATCH_SIZE,
    shuffle: bool = False,
) -> DataLoader:
    if model_input == "current_day":
        x = arrays["X_mlp_current_day"]
    elif model_input == "flattened":
        x = arrays["X_mlp_flattened"]
    elif model_input == "sequence":
        x = arrays["X_sequence"]
    else:
        raise ValueError(f"Unknown model_input: {model_input}")

    y = arrays["y"].astype(np.float32)
    x_tensor = torch.tensor(x, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    dataset = TensorDataset(x_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def make_loaders(window: int, model_input: str) -> Dict[str, DataLoader]:
    data = load_window_data(window)
    return {
        "train": make_loader(data["train"], model_input, shuffle=True),
        "validation": make_loader(data["validation"], model_input, shuffle=False),
        "test": make_loader(data["test"], model_input, shuffle=False),
    }


In [8]:
# 6. 모델 정의
# 모든 모델은 binary classification용 logit 1개를 출력합니다. BCEWithLogitsLoss가 내부적으로 sigmoid를 처리합니다.


class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, dropout: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class RecurrentClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        rnn_type: str,
        hidden_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.2,
        bidirectional: bool = False,
    ):
        super().__init__()
        self.rnn_type = rnn_type
        recurrent_dropout = dropout if num_layers > 1 else 0.0

        if rnn_type == "SimpleRNN":
            rnn_cls = nn.RNN
        elif rnn_type == "LSTM":
            rnn_cls = nn.LSTM
        elif rnn_type == "GRU":
            rnn_cls = nn.GRU
        else:
            raise ValueError(f"Unsupported rnn_type: {rnn_type}")

        self.rnn = rnn_cls(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
            bidirectional=bidirectional,
        )
        direction_factor = 2 if bidirectional else 1
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * direction_factor, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output, _ = self.rnn(x)
        last_step = output[:, -1, :]
        return self.head(last_step)


class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim: int, channels: int = 64, dropout: float = 0.25):
        super().__init__()
        self.features = nn.Sequential(
            # Conv1d는 batch x channels/features x time_steps 형태를 기대합니다.
            nn.Conv1d(input_dim, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(channels),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.head = nn.Linear(channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.features(x).squeeze(-1)
        return self.head(x)


In [9]:
# 7. 모델 생성 함수
# 나중에 architecture tuning을 하고 싶으면 여기서 hidden_dim/dropout 등을 조정하세요. 처음에는 작은 고정 모델로 비교합니다.



def infer_input_dims(window: int) -> dict:
    train = load_split_npz(window, "train")
    return {
        "current_day_dim": train["X_mlp_current_day"].shape[1],
        "flattened_dim": train["X_mlp_flattened"].shape[1],
        "sequence_steps": train["X_sequence"].shape[1],
        "sequence_features": train["X_sequence"].shape[2],
    }


def build_model(model_name: str, window: int) -> Tuple[nn.Module, str]:
    dims = infer_input_dims(window)

    if model_name == "mlp_current_day":
        return MLPClassifier(dims["current_day_dim"]), "current_day"
    if model_name == "mlp_flattened":
        return MLPClassifier(dims["flattened_dim"]), "flattened"
    if model_name == "SimpleRNN":
        return RecurrentClassifier(dims["sequence_features"], rnn_type="SimpleRNN"), "sequence"
    if model_name == "LSTM":
        return RecurrentClassifier(dims["sequence_features"], rnn_type="LSTM"), "sequence"
    if model_name == "GRU":
        return RecurrentClassifier(dims["sequence_features"], rnn_type="GRU"), "sequence"
    if model_name == "BiLSTM":
        return RecurrentClassifier(dims["sequence_features"], rnn_type="LSTM", bidirectional=True), "sequence"
    if model_name == "CNN1D":
        return CNN1DClassifier(dims["sequence_features"]), "sequence"

    raise ValueError(f"Unknown model_name: {model_name}")

MODEL_NAMES = [
    "mlp_current_day",
    "mlp_flattened",
    "SimpleRNN",
    "LSTM",
    "GRU",
    "BiLSTM",
    "CNN1D",
]


In [10]:
# 8. Metric 계산 함수
# balanced_accuracy를 primary metric으로 사용하고, 보조 지표로 trade-off를 확인합니다.



def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_true = y_true.astype(int)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "average_precision": average_precision_score(y_true, y_prob),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        metrics["roc_auc"] = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics.update({"tn": tn, "fp": fp, "fn": fn, "tp": tp})
    return metrics


@torch.no_grad()
def predict_probabilities(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs = []
    targets = []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)
        prob = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        probs.append(prob)
        targets.append(y.numpy().ravel())
    return np.concatenate(targets), np.concatenate(probs)


In [11]:
# 9. 학습 루프
# 현재 early stopping은 validation loss 기준입니다. 필요하면 나중에 validation balanced accuracy 기준으로 바꿀 수 있습니다.


@dataclass
class ExperimentConfig:
    model_name: str
    window: int
    max_epochs: int = MAX_EPOCHS
    batch_size: int = BATCH_SIZE
    learning_rate: float = LEARNING_RATE
    weight_decay: float = WEIGHT_DECAY
    patience: int = PATIENCE
    seed: int = RANDOM_STATE


def train_one_experiment(config: ExperimentConfig) -> Tuple[nn.Module, pd.DataFrame, dict]:
    set_seed(config.seed)
    model, model_input = build_model(config.model_name, config.window)
    model = model.to(DEVICE)
    loaders = make_loaders(config.window, model_input)

    train_arrays = load_split_npz(config.window, "train")
    y_train = train_arrays["y"]
    positives = float((y_train == 1).sum())
    negatives = float((y_train == 0).sum())
    pos_weight = torch.tensor([negatives / max(positives, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )

    best_state = None
    best_val_loss = math.inf
    best_epoch = 0
    patience_left = config.patience
    history_rows = []

    for epoch in range(1, config.max_epochs + 1):
        model.train()
        train_losses = []
        for x, y in loaders["train"]:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x, y in loaders["validation"]:
                x = x.to(DEVICE)
                y = y.to(DEVICE)
                logits = model(x)
                loss = criterion(logits, y)
                val_losses.append(float(loss.detach().cpu()))

        y_val, p_val = predict_probabilities(model, loaders["validation"])
        val_metrics = compute_metrics(y_val, p_val)
        row = {
            **asdict(config),
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "validation_loss": float(np.mean(val_losses)),
            **{f"validation_{k}": v for k, v in val_metrics.items()},
        }
        history_rows.append(row)

        if row["validation_loss"] < best_val_loss:
            best_val_loss = row["validation_loss"]
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = config.patience
        else:
            patience_left -= 1

        if patience_left <= 0:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        model = model.to(DEVICE)

    history = pd.DataFrame(history_rows)
    summary = {
        **asdict(config),
        "model_input": model_input,
        "best_epoch": best_epoch,
        "best_validation_loss": best_val_loss,
    }
    return model, history, summary


In [12]:
# 10. 평가 및 저장 함수
# 각 run은 나중의 비교, calibration, bootstrap 확인을 위해 validation/test prediction을 저장합니다.



def evaluate_model(model: nn.Module, config: ExperimentConfig, model_input: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    loaders = make_loaders(config.window, model_input)
    metric_rows = []
    prediction_rows = []

    window_data = load_window_data(config.window)
    for split in ["validation", "test"]:
        y_true, y_prob = predict_probabilities(model, loaders[split])
        metrics = compute_metrics(y_true, y_prob)
        metric_rows.append({
            **asdict(config),
            "split": split,
            **metrics,
        })

        metadata = window_data[split]
        for idx in range(len(y_true)):
            prediction_rows.append({
                **asdict(config),
                "split": split,
                "row_index": idx,
                "participant_object_id": str(metadata["participant_object_id"][idx]),
                "window_start_date": str(metadata["window_start_date"][idx]),
                "window_end_date": str(metadata["window_end_date"][idx]),
                "y_true": int(y_true[idx]),
                "y_prob": float(y_prob[idx]),
                "y_pred": int(y_prob[idx] >= 0.5),
            })

    return pd.DataFrame(metric_rows), pd.DataFrame(prediction_rows)


def run_and_save_experiment(config: ExperimentConfig) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    model, history, summary = train_one_experiment(config)
    _, model_input = build_model(config.model_name, config.window)
    metrics, predictions = evaluate_model(model, config, model_input)

    run_id = f"{config.model_name}_window_{config.window}_seed_{config.seed}"
    torch.save(model.state_dict(), MODEL_DIR / f"{run_id}.pt")
    history.to_csv(OUTPUT_DIR / f"{run_id}_history.csv", index=False, encoding="utf-8-sig")
    predictions.to_csv(OUTPUT_DIR / f"{run_id}_predictions.csv", index=False, encoding="utf-8-sig")
    (OUTPUT_DIR / f"{run_id}_config.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )
    return metrics, predictions, history


In [13]:
# 11. Smoke test
# 전체 실험 그리드 실행 전에 이 셀을 먼저 돌려 학습 루프와 tensor shape를 확인하세요.


if RUN_SMOKE_TEST:
    smoke_config = ExperimentConfig(
        model_name="GRU",
        window=7,
        max_epochs=2,
        patience=1,
    )
    smoke_metrics, smoke_predictions, smoke_history = run_and_save_experiment(smoke_config)
    display(smoke_metrics)
    display(smoke_history.tail())
else:
    print("RUN_SMOKE_TEST=False이므로 smoke test를 건너뜁니다.")


,model_name,window,max_epochs,batch_size,learning_rate,weight_decay,patience,seed,split,accuracy,balanced_accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp
0,GRU,7,2,64,0.001,0.0001,1,42,validation,0.617100,0.664255,0.508108,0.886792,0.646048,0.589521,0.723348,72,91,12,94
1,GRU,7,2,64,0.001,0.0001,1,42,test,0.541401,0.516983,0.444444,0.363636,0.400000,0.480766,0.500791,122,60,84,48


,model_name,window,max_epochs,batch_size,learning_rate,weight_decay,patience,seed,epoch,train_loss,...,validation_balanced_accuracy,validation_precision,validation_recall,validation_f1,validation_average_precision,validation_roc_auc,validation_tn,validation_fp,validation_fn,validation_tp
0,GRU,7,2,64,0.001,0.0001,1,42,1,0.808122,...,0.601516,0.468571,0.773585,0.583630,0.524325,0.653432,70,93,24,82
1,GRU,7,2,64,0.001,0.0001,1,42,2,0.716380,...,0.664255,0.508108,0.886792,0.646048,0.589521,0.723348,72,91,12,94


In [21]:
# 12. 전체 1차 실험 그리드
# smoke test가 성공하면 1번 셀에서 RUN_FULL_EXPERIMENTS=True로 바꾸세요. 최소 21개 실험이 실행됩니다.


all_metric_frames = []
all_prediction_frames = []
all_history_frames = []

if RUN_FULL_EXPERIMENTS:
    for window in WINDOWS:
        for model_name in MODEL_NAMES:
            print(f"Running model={model_name}, window={window}")
            config = ExperimentConfig(model_name=model_name, window=window)
            metrics, predictions, history = run_and_save_experiment(config)
            all_metric_frames.append(metrics)
            all_prediction_frames.append(predictions)
            all_history_frames.append(history)

    all_metrics = pd.concat(all_metric_frames, ignore_index=True)
    all_predictions = pd.concat(all_prediction_frames, ignore_index=True)
    all_history = pd.concat(all_history_frames, ignore_index=True)

    all_metrics.to_csv(OUTPUT_DIR / "deep_learning_model_metrics.csv", index=False, encoding="utf-8-sig")
    all_predictions.to_csv(OUTPUT_DIR / "deep_learning_model_predictions.csv", index=False, encoding="utf-8-sig")
    all_history.to_csv(OUTPUT_DIR / "deep_learning_training_history.csv", index=False, encoding="utf-8-sig")

    display(all_metrics.sort_values(["split", "balanced_accuracy"], ascending=[True, False]))
else:
    print("RUN_FULL_EXPERIMENTS=False입니다. 전체 실험을 실행하려면 1번 셀에서 True로 바꾸세요.")


Running model=mlp_current_day, window=7
Running model=mlp_flattened, window=7
Running model=SimpleRNN, window=7
Running model=LSTM, window=7
Running model=GRU, window=7
Running model=BiLSTM, window=7
Running model=CNN1D, window=7
Running model=mlp_current_day, window=14
Running model=mlp_flattened, window=14
Running model=SimpleRNN, window=14
Running model=LSTM, window=14
Running model=GRU, window=14
Running model=BiLSTM, window=14
Running model=CNN1D, window=14
Running model=mlp_current_day, window=30
Running model=mlp_flattened, window=30
Running model=SimpleRNN, window=30
Running model=LSTM, window=30
Running model=GRU, window=30
Running model=BiLSTM, window=30
Running model=CNN1D, window=30


,model_name,window,max_epochs,batch_size,learning_rate,weight_decay,patience,seed,split,accuracy,balanced_accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp
9,GRU,7,40,64,0.001,0.0001,6,42,test,0.671975,0.646270,0.646465,0.484848,0.554113,0.666109,0.733392,147,35,68,64
1,mlp_current_day,7,40,64,0.001,0.0001,6,42,test,0.646497,0.634699,0.582677,0.560606,0.571429,0.613601,0.644772,129,53,58,74
25,BiLSTM,14,40,64,0.001,0.0001,6,42,test,0.626168,0.610046,0.640625,0.418367,0.506173,0.627513,0.635556,93,23,57,41
5,SimpleRNN,7,40,64,0.001,0.0001,6,42,test,0.636943,0.609807,0.591837,0.439394,0.504348,0.603490,0.685440,142,40,74,58
29,mlp_current_day,30,40,64,0.001,0.0001,6,42,test,0.575472,0.594869,0.704545,0.492063,0.579439,0.735673,0.629014,30,13,32,31
23,GRU,14,40,64,0.001,0.0001,6,42,test,0.602804,0.594828,0.576471,0.500000,0.535519,0.603781,0.624472,80,36,49,49
11,BiLSTM,7,40,64,0.001,0.0001,6,42,test,0.624204,0.579046,0.609375,0.295455,0.397959,0.544376,0.589244,157,25,93,39
13,CNN1D,7,40,64,0.001,0.0001,6,42,test,0.627389,0.575549,0.647059,0.250000,0.360656,0.577988,0.593365,164,18,99,33
39,BiLSTM,30,40,64,0.001,0.0001,6,42,test,0.518868,0.554633,0.676471,0.365079,0.474227,0.691856,0.577704,32,11,40,23
3,mlp_flattened,7,40,64,0.001,0.0001,6,42,test,0.576433,0.551365,0.495238,0.393939,0.438819,0.494551,0.540709,129,53,80,52


In [22]:
# 13. Validation 기준 순위 확인
# 전체 실험이 끝난 뒤 실행하세요. 모델 선택은 validation 결과를 기준으로 합니다.


metrics_path = OUTPUT_DIR / "deep_learning_model_metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    validation_rank = (
        metrics[metrics["split"] == "validation"]
        .sort_values(["balanced_accuracy", "roc_auc", "f1"], ascending=False)
        .reset_index(drop=True)
    )
    display(validation_rank)
else:
    print(f"아직 metric 파일이 없습니다: {metrics_path}")


,model_name,window,max_epochs,batch_size,learning_rate,weight_decay,patience,seed,split,accuracy,balanced_accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp
0,BiLSTM,30,40,64,0.001,0.0001,6,42,validation,0.836364,0.866300,0.600000,0.923077,0.727273,0.678372,0.915751,34,8,1,12
1,LSTM,30,40,64,0.001,0.0001,6,42,validation,0.836364,0.866300,0.600000,0.923077,0.727273,0.620632,0.897436,34,8,1,12
2,GRU,30,40,64,0.001,0.0001,6,42,validation,0.818182,0.854396,0.571429,0.923077,0.705882,0.637715,0.886447,33,9,1,12
3,BiLSTM,14,40,64,0.001,0.0001,6,42,validation,0.828767,0.850258,0.691176,0.921569,0.789916,0.757053,0.886687,74,21,4,47
4,mlp_current_day,30,40,64,0.001,0.0001,6,42,validation,0.836364,0.839744,0.611111,0.846154,0.709677,0.703245,0.902930,35,7,2,11
5,GRU,14,40,64,0.001,0.0001,6,42,validation,0.801370,0.824665,0.657143,0.901961,0.760331,0.715264,0.863777,71,24,5,46
6,SimpleRNN,30,40,64,0.001,0.0001,6,42,validation,0.763636,0.818681,0.500000,0.923077,0.648649,0.740010,0.899267,30,12,1,12
7,CNN1D,30,40,64,0.001,0.0001,6,42,validation,0.800000,0.815934,0.550000,0.846154,0.666667,0.696917,0.879121,33,9,2,11
8,LSTM,14,40,64,0.001,0.0001,6,42,validation,0.787671,0.786894,0.666667,0.784314,0.720721,0.735551,0.864396,75,20,11,40
9,mlp_current_day,7,40,64,0.001,0.0001,6,42,validation,0.762082,0.782238,0.645833,0.877358,0.744000,0.790948,0.865609,112,51,13,93


In [23]:
# 14. Test 결과 확인
# test 결과는 validation 기반 모델 선택이 끝난 뒤 최종 확인용으로만 사용하세요. test 결과로 튜닝하면 안 됩니다.


if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    test_rank = (
        metrics[metrics["split"] == "test"]
        .sort_values(["balanced_accuracy", "roc_auc", "f1"], ascending=False)
        .reset_index(drop=True)
    )
    display(test_rank)
else:
    print(f"아직 metric 파일이 없습니다: {metrics_path}")


,model_name,window,max_epochs,batch_size,learning_rate,weight_decay,patience,seed,split,accuracy,balanced_accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp
0,GRU,7,40,64,0.001,0.0001,6,42,test,0.671975,0.646270,0.646465,0.484848,0.554113,0.666109,0.733392,147,35,68,64
1,mlp_current_day,7,40,64,0.001,0.0001,6,42,test,0.646497,0.634699,0.582677,0.560606,0.571429,0.613601,0.644772,129,53,58,74
2,BiLSTM,14,40,64,0.001,0.0001,6,42,test,0.626168,0.610046,0.640625,0.418367,0.506173,0.627513,0.635556,93,23,57,41
3,SimpleRNN,7,40,64,0.001,0.0001,6,42,test,0.636943,0.609807,0.591837,0.439394,0.504348,0.603490,0.685440,142,40,74,58
4,mlp_current_day,30,40,64,0.001,0.0001,6,42,test,0.575472,0.594869,0.704545,0.492063,0.579439,0.735673,0.629014,30,13,32,31
5,GRU,14,40,64,0.001,0.0001,6,42,test,0.602804,0.594828,0.576471,0.500000,0.535519,0.603781,0.624472,80,36,49,49
6,BiLSTM,7,40,64,0.001,0.0001,6,42,test,0.624204,0.579046,0.609375,0.295455,0.397959,0.544376,0.589244,157,25,93,39
7,CNN1D,7,40,64,0.001,0.0001,6,42,test,0.627389,0.575549,0.647059,0.250000,0.360656,0.577988,0.593365,164,18,99,33
8,BiLSTM,30,40,64,0.001,0.0001,6,42,test,0.518868,0.554633,0.676471,0.365079,0.474227,0.691856,0.577704,32,11,40,23
9,mlp_flattened,7,40,64,0.001,0.0001,6,42,test,0.576433,0.551365,0.495238,0.393939,0.438819,0.494551,0.540709,129,53,80,52


In [24]:
# 15. 전통 ML reference baseline 비교
# 딥러닝 후보 모델을 wearable_only + Logistic Regression 기준선과 비교합니다.


REFERENCE_BASELINE = {
    "model_name": "traditional_ml_reference__wearable_only_logistic_regression",
    "split": "test",
    "balanced_accuracy": 0.8076,
    "roc_auc": 0.9046,
    "f1": 0.7556,
    "precision": 0.6996,
    "recall": 0.8214,
}

if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    test_metrics = metrics[metrics["split"] == "test"].copy()
    comparison_columns = [
        "model_name",
        "window",
        "split",
        "balanced_accuracy",
        "roc_auc",
        "f1",
        "precision",
        "recall",
    ]
    baseline_row = pd.DataFrame([{**REFERENCE_BASELINE, "window": "-"}])
    comparison = pd.concat(
        [test_metrics[comparison_columns], baseline_row[comparison_columns]],
        ignore_index=True,
    ).sort_values("balanced_accuracy", ascending=False)
    display(comparison)
else:
    print(f"아직 metric 파일이 없습니다: {metrics_path}")


,model_name,window,split,balanced_accuracy,roc_auc,f1,precision,recall
21,traditional_ml_reference__wearable_only_logist...,-,test,0.807600,0.904600,0.755600,0.699600,0.821400
4,GRU,7,test,0.646270,0.733392,0.554113,0.646465,0.484848
0,mlp_current_day,7,test,0.634699,0.644772,0.571429,0.582677,0.560606
12,BiLSTM,14,test,0.610046,0.635556,0.506173,0.640625,0.418367
2,SimpleRNN,7,test,0.609807,0.685440,0.504348,0.591837,0.439394
14,mlp_current_day,30,test,0.594869,0.629014,0.579439,0.704545,0.492063
11,GRU,14,test,0.594828,0.624472,0.535519,0.576471,0.500000
5,BiLSTM,7,test,0.579046,0.589244,0.397959,0.609375,0.295455
6,CNN1D,7,test,0.575549,0.593365,0.360656,0.647059,0.250000
19,BiLSTM,30,test,0.554633,0.577704,0.474227,0.676471,0.365079


In [25]:
# 16. Participant-level bootstrap 기본 코드
# validation 기준으로 후보를 선택한 뒤 해당 후보의 test prediction에 대해 실행하세요.


SELECTED_MODEL_NAME = "GRU"      # validation 기준으로 선택한 모델명으로 바꾸세요.
SELECTED_WINDOW = 7              # validation 기준으로 선택한 window로 바꾸세요.
BOOTSTRAP_REPEATS = 1000


def participant_bootstrap(predictions: pd.DataFrame, repeats: int = BOOTSTRAP_REPEATS) -> pd.DataFrame:
    participants = predictions["participant_object_id"].unique()
    rows = []
    rng = np.random.default_rng(RANDOM_STATE)

    for repeat in range(repeats):
        sampled = rng.choice(participants, size=len(participants), replace=True)
        sampled_df = pd.concat(
            [predictions[predictions["participant_object_id"] == pid] for pid in sampled],
            ignore_index=True,
        )
        if sampled_df["y_true"].nunique() < 2:
            continue
        metric_row = compute_metrics(
            sampled_df["y_true"].to_numpy(),
            sampled_df["y_prob"].to_numpy(),
        )
        metric_row["repeat"] = repeat
        rows.append(metric_row)
    return pd.DataFrame(rows)

predictions_path = OUTPUT_DIR / "deep_learning_model_predictions.csv"
if predictions_path.exists():
    predictions = pd.read_csv(predictions_path)
    selected_predictions = predictions[
        (predictions["split"] == "test")
        & (predictions["model_name"] == SELECTED_MODEL_NAME)
        & (predictions["window"] == SELECTED_WINDOW)
    ].copy()

    if selected_predictions.empty:
        print("선택한 model/window에 해당하는 prediction이 없습니다. SELECTED_MODEL_NAME과 SELECTED_WINDOW를 확인하세요.")
    else:
        bootstrap = participant_bootstrap(selected_predictions)
        display(bootstrap.describe(percentiles=[0.025, 0.5, 0.975]).T)
        bootstrap.to_csv(
            OUTPUT_DIR / f"{SELECTED_MODEL_NAME}_window_{SELECTED_WINDOW}_participant_bootstrap.csv",
            index=False,
            encoding="utf-8-sig",
        )
else:
    print(f"아직 prediction 파일이 없습니다: {predictions_path}")


,count,mean,std,min,2.5%,50%,97.5%,max
accuracy,1000.0,0.672452,0.047293,0.515244,0.570379,0.675428,0.746315,0.773077
balanced_accuracy,1000.0,0.650317,0.049426,0.515057,0.538944,0.651977,0.717264,0.739899
precision,1000.0,0.632297,0.100418,0.225806,0.406780,0.646154,0.788742,0.839416
recall,1000.0,0.499211,0.128018,0.121951,0.231446,0.496589,0.705128,0.837838
f1,1000.0,0.545610,0.094935,0.198020,0.318940,0.560606,0.685041,0.711370
average_precision,1000.0,0.652830,0.109082,0.271066,0.391676,0.677098,0.798990,0.839635
roc_auc,1000.0,0.737157,0.049746,0.590128,0.627023,0.744476,0.808700,0.873813
tn,1000.0,145.805000,38.356280,41.000000,74.975000,144.000000,222.000000,276.000000
fp,1000.0,35.069000,9.333561,9.000000,17.000000,35.000000,54.000000,64.000000
fn,1000.0,67.991000,31.822438,5.000000,16.975000,65.000000,141.025000,192.000000


In [26]:
# 17. 공유용 결과 요약
# 이 표와 CSV 결과를 알려주시면 해석과 보고서 작성을 이어갈 수 있습니다.


if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    print("Metrics 저장 위치:", metrics_path)
    print("Predictions 저장 위치:", OUTPUT_DIR / "deep_learning_model_predictions.csv")
    print("Training history 저장 위치:", OUTPUT_DIR / "deep_learning_training_history.csv")
    print("\nValidation 상위 10개:")
    display(
        metrics[metrics["split"] == "validation"]
        .sort_values(["balanced_accuracy", "roc_auc", "f1"], ascending=False)
        .head(10)
    )
    print("\nTest 상위 10개, validation 기반 선택 이후에만 확인하세요:")
    display(
        metrics[metrics["split"] == "test"]
        .sort_values(["balanced_accuracy", "roc_auc", "f1"], ascending=False)
        .head(10)
    )
else:
    print("아직 전체 실험 결과가 없습니다. RUN_FULL_EXPERIMENTS=True로 바꾼 뒤 전체 그리드를 먼저 실행하세요.")


Metrics 저장 위치: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_experiments\deep_learning_model_metrics.csv
Predictions 저장 위치: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_experiments\deep_learning_model_predictions.csv
Training history 저장 위치: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_experiments\deep_learning_training_history.csv

Validation 상위 10개:


,model_name,window,max_epochs,batch_size,learning_rate,weight_decay,patience,seed,split,accuracy,balanced_accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp
38,BiLSTM,30,40,64,0.001,0.0001,6,42,validation,0.836364,0.866300,0.600000,0.923077,0.727273,0.678372,0.915751,34,8,1,12
34,LSTM,30,40,64,0.001,0.0001,6,42,validation,0.836364,0.866300,0.600000,0.923077,0.727273,0.620632,0.897436,34,8,1,12
36,GRU,30,40,64,0.001,0.0001,6,42,validation,0.818182,0.854396,0.571429,0.923077,0.705882,0.637715,0.886447,33,9,1,12
24,BiLSTM,14,40,64,0.001,0.0001,6,42,validation,0.828767,0.850258,0.691176,0.921569,0.789916,0.757053,0.886687,74,21,4,47
28,mlp_current_day,30,40,64,0.001,0.0001,6,42,validation,0.836364,0.839744,0.611111,0.846154,0.709677,0.703245,0.902930,35,7,2,11
22,GRU,14,40,64,0.001,0.0001,6,42,validation,0.801370,0.824665,0.657143,0.901961,0.760331,0.715264,0.863777,71,24,5,46
32,SimpleRNN,30,40,64,0.001,0.0001,6,42,validation,0.763636,0.818681,0.500000,0.923077,0.648649,0.740010,0.899267,30,12,1,12
40,CNN1D,30,40,64,0.001,0.0001,6,42,validation,0.800000,0.815934,0.550000,0.846154,0.666667,0.696917,0.879121,33,9,2,11
20,LSTM,14,40,64,0.001,0.0001,6,42,validation,0.787671,0.786894,0.666667,0.784314,0.720721,0.735551,0.864396,75,20,11,40
0,mlp_current_day,7,40,64,0.001,0.0001,6,42,validation,0.762082,0.782238,0.645833,0.877358,0.744000,0.790948,0.865609,112,51,13,93



Test 상위 10개, validation 기반 선택 이후에만 확인하세요:


,model_name,window,max_epochs,batch_size,learning_rate,weight_decay,patience,seed,split,accuracy,balanced_accuracy,precision,recall,f1,average_precision,roc_auc,tn,fp,fn,tp
9,GRU,7,40,64,0.001,0.0001,6,42,test,0.671975,0.646270,0.646465,0.484848,0.554113,0.666109,0.733392,147,35,68,64
1,mlp_current_day,7,40,64,0.001,0.0001,6,42,test,0.646497,0.634699,0.582677,0.560606,0.571429,0.613601,0.644772,129,53,58,74
25,BiLSTM,14,40,64,0.001,0.0001,6,42,test,0.626168,0.610046,0.640625,0.418367,0.506173,0.627513,0.635556,93,23,57,41
5,SimpleRNN,7,40,64,0.001,0.0001,6,42,test,0.636943,0.609807,0.591837,0.439394,0.504348,0.603490,0.685440,142,40,74,58
29,mlp_current_day,30,40,64,0.001,0.0001,6,42,test,0.575472,0.594869,0.704545,0.492063,0.579439,0.735673,0.629014,30,13,32,31
23,GRU,14,40,64,0.001,0.0001,6,42,test,0.602804,0.594828,0.576471,0.500000,0.535519,0.603781,0.624472,80,36,49,49
11,BiLSTM,7,40,64,0.001,0.0001,6,42,test,0.624204,0.579046,0.609375,0.295455,0.397959,0.544376,0.589244,157,25,93,39
13,CNN1D,7,40,64,0.001,0.0001,6,42,test,0.627389,0.575549,0.647059,0.250000,0.360656,0.577988,0.593365,164,18,99,33
39,BiLSTM,30,40,64,0.001,0.0001,6,42,test,0.518868,0.554633,0.676471,0.365079,0.474227,0.691856,0.577704,32,11,40,23
3,mlp_flattened,7,40,64,0.001,0.0001,6,42,test,0.576433,0.551365,0.495238,0.393939,0.438819,0.494551,0.540709,129,53,80,52
